# UTS PjBL Kecerdasan Buatan & Sistem Cerdas — TIM 12
## Bot Game AI Connect-Four Berbasis Depth-Limited Minimax & Evaluation Function

- **Institusi:** Institut Digital Ekonomi (IDE) LPKIA Bandung — S1 Teknik Informatika / Sistem Informasi
- **Dosen Pengampu:** Wahyu Nurjaya WK, S.T., M.Kom.
- **Ruang Lingkup & Metode AI:** *Adversarial Search* (*Depth-Limited Minimax* + *Heuristic Evaluation Function* & *Alpha-Beta Pruning*)
- **Demo Web UI (GitHub Pages):** [https://zysrnh.github.io/ConnectFour_UTS/](https://zysrnh.github.io/ConnectFour_UTS/)
- **Repository GitHub:** [https://github.com/zysrnh/ConnectFour_UTS](https://github.com/zysrnh/ConnectFour_UTS)

### Identitas TIM 12
| Peran | Nama Mahasiswa | NIM | Tanggung Jawab Utama |
| :--- | :--- | :--- | :--- |
| **Mahasiswa A** | `[Isi Nama Mahasiswa A]` | `[NIM A]` | *AI System Architect / Modeler* (State Space, Eval Function, Minimax) |
| **Mahasiswa B** | `[Isi Nama Mahasiswa B]` | `[NIM B]` | *Lead Developer / Tester* (Arena Interaktif, Benchmark, Web UI) |

---
## Bagian 1: Pemodelan AI & Relevansi dengan Materi Sesi 1–4

Implementasi proyek ini memadukan konsep inti yang telah dipelajari dari **Sesi 1 hingga Sesi 4**:

1. **Arsitektur Sistem Cerdas & Intelligent Agent (Sesi 1):**
   - **Input:** Status papan Connect-Four `6 × 7` saat ini dan aksi kolom pemain.
   - **Knowledge Base:** Aturan kemenangan `4-in-a-row` dan bobot fitur *Evaluation Function* (`EVAL_WEIGHTS`).
   - **Inference Engine:** Algoritma *Depth-Limited Minimax* (serta opsi optimasi *Alpha-Beta Pruning*).
   - **Output:** Kolom terbaik `[0..6]`, skor evaluasi papan, dan jumlah node yang dievaluasi.
   - **Formulasi Agen:** `Percept` berupa posisi papan `6 × 7`; `Action` menjatuhkan bidak ke kolom valid; `Performance Measure` diukur dari keberhasilan menang (`WIN_SCORE`), pemblokiran ancaman lawan, dan efisiensi node.
2. **State Space & Search Tree (Sesi 2):**
   - Ruang keadaan dimodelkan sebagai *Game Tree* dengan *branching factor* `b <= 7` dan kedalaman maksimal permainan `m = 42`.
   - Penelusuran dilakukan secara rekursif berbasis *Depth-First Search* (DFS).
3. **Heuristic Function `h(n)` (Sesi 3):**
   - Ketika batas kedalaman pencarian tercapai pada posisi non-terminal (`depth >= max_depth`), fungsi `evaluate_board()` bertindak sebagai fungsi heuristik untuk menaksir keunggulan papan bagi Bot.
4. **Adversarial Search: Minimax & Alpha-Beta Pruning (Sesi 4):**
   - Permainan bersifat *zero-sum*: **Bot AI (`2` / `X`) sebagai MAX** dan **Manusia (`1` / `O`) sebagai MIN**.
   - **Kenapa pakai Depth-Limited Minimax?** Seperti dibahas pada Pre-Test Sesi 4, Minimax murni memiliki kompleksitas `O(b^m)`. Pada Connect-Four (`b <= 7, m = 42`), pencarian penuh sampai terminal state tidak efisien. Pembatasan kedalaman (`max_depth`) + *Evaluation Function* membuat keputusan cepat dan akurat.
   - **Optimasi Alpha-Beta Pruning:** Memangkas cabang yang tidak memengaruhi hasil akhir saat `alpha >= beta`, didukung *center-first column ordering* `[3, 2, 4, 1, 5, 0, 6]`.

In [ ]:
# Setup Environment & Konstanta Global (Sinkron 1:1 dengan js/ai.js)
import math
import random
import sys
import time
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

ROWS = 6
COLS = 7
WINDOW_LENGTH = 4

EMPTY = 0
HUMAN = 1   # Pemain 1 / Manusia (MIN) -> Simbol 'O'
AI = 2      # Pemain 2 / Bot AI (MAX)   -> Simbol 'X'

WIN_SCORE = 100000
COLUMN_ORDER = [3, 2, 4, 1, 5, 0, 6]  # Prioritas kolom tengah ke pinggir

EVAL_WEIGHTS = {
    "WIN": 100000,
    "AI_THREE": 12,
    "AI_TWO": 3,
    "OPP_THREE": -80,
    "OPP_TWO": -2,
    "CENTER_PER_DISC": 4,
}

DIFFICULTY_PRESETS = {
    "Easy": {"max_depth": 0, "use_pruning": False, "label": "Easy (Random Move)"},
    "Medium": {"max_depth": 3, "use_pruning": False, "label": "Medium (Depth-Limited Minimax, depth=3)"},
    "Hard": {"max_depth": 5, "use_pruning": True, "label": "Hard (Depth-Limited + Alpha-Beta, depth=5)"},
}

print("=== CEK ENVIRONMENT PRAKTIKUM AI TIM 12 ===")
print("Versi Python       :", sys.version.split()[0])
print("Matplotlib tersedia:", HAS_MATPLOTLIB)
print("Konfigurasi Papan  :", f"{ROWS} baris x {COLS} kolom")

---
## Bagian 2: Representasi Papan & Aturan Permainan Connect-Four
Papan direpresentasikan sebagai matriks 2D berukuran `6 × 7` (`row 0` = baris paling atas, `row 5` = baris paling bawah). Setiap bidak yang dijatuhkan ke kolom akan jatuh ke baris kosong terbawah akibat gravitasi.

In [ ]:
def create_board():
    """Buat papan kosong 6x7 (baris 0 = atas, baris 5 = bawah)."""
    return [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]


def clone_board(board):
    """Salin papan 2D secara mendalam."""
    return [row[:] for row in board]


def print_board(board):
    """Tampilkan papan Connect-Four ke konsol dengan label kolom 0-6."""
    symbol_map = {EMPTY: ".", HUMAN: "O", AI: "X"}
    print("  0 1 2 3 4 5 6")
    print(" +" + "--" * COLS + "+")
    for r in range(ROWS):
        row_str = " ".join(symbol_map[board[r][c]] for c in range(COLS))
        print(f"{r}| {row_str}|")
    print(" +" + "--" * COLS + "+")


def is_valid_location(board, col):
    """Cek apakah kolom valid dan baris paling atas masih kosong."""
    return 0 <= col < COLS and board[0][col] == EMPTY


def get_valid_locations(board):
    """Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir."""
    return [col for col in COLUMN_ORDER if is_valid_location(board, col)]


def get_next_open_row(board, col):
    """Cari indeks baris terbawah yang masih kosong pada kolom tertentu."""
    for r in range(ROWS - 1, -1, -1):
        if board[r][col] == EMPTY:
            return r
    return -1


def drop_piece(board, row, col, piece):
    """Letakkan bidak pada koordinat (row, col)."""
    board[row][col] = piece


def get_winner_info(board):
    """Cek pemenang (4-in-a-row) atau seri beserta daftar koordinat kemenangan."""
    # Cek horizontal
    for r in range(ROWS):
        for c in range(COLS - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p != EMPTY and p == board[r][c + 1] == board[r][c + 2] == board[r][c + 3]:
                return p, [(r, c), (r, c + 1), (r, c + 2), (r, c + 3)]

    # Cek vertikal
    for c in range(COLS):
        for r in range(ROWS - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p != EMPTY and p == board[r + 1][c] == board[r + 2][c] == board[r + 3][c]:
                return p, [(r, c), (r + 1, c), (r + 2, c), (r + 3, c)]

    # Cek diagonal turun ke kanan (\)
    for r in range(ROWS - WINDOW_LENGTH + 1):
        for c in range(COLS - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p != EMPTY and p == board[r + 1][c + 1] == board[r + 2][c + 2] == board[r + 3][c + 3]:
                return p, [(r, c), (r + 1, c + 1), (r + 2, c + 2), (r + 3, c + 3)]

    # Cek diagonal naik ke kanan (/)
    for r in range(WINDOW_LENGTH - 1, ROWS):
        for c in range(COLS - WINDOW_LENGTH + 1):
            p = board[r][c]
            if p != EMPTY and p == board[r - 1][c + 1] == board[r - 2][c + 2] == board[r - 3][c + 3]:
                return p, [(r, c), (r - 1, c + 1), (r - 2, c + 2), (r - 3, c + 3)]

    # Cek seri jika baris paling atas sudah penuh semua
    if all(board[0][c] != EMPTY for c in range(COLS)):
        return "DRAW", []

    return None, []


def check_winner(board):
    """Wrapper ringkas mengembalikan 1 (HUMAN), 2 (AI), 'DRAW', atau None."""
    winner, _ = get_winner_info(board)
    return winner


# Uji cepat papan & cek kemenangan
sample_board = create_board()
for col_idx in [3, 3, 2, 2, 4, 1]:
    r_open = get_next_open_row(sample_board, col_idx)
    drop_piece(sample_board, r_open, col_idx, AI if col_idx != 1 else HUMAN)

print_board(sample_board)
print("Status Game Contoh:", check_winner(sample_board))

---
## Bagian 3: Fungsi Evaluasi Papan (*Heuristic Evaluation Function*)
Karena pencarian dibatasi oleh `max_depth`, Bot AI membutuhkan **Evaluation Function** untuk menilai posisi papan non-terminal:

- Memeriksa seluruh **69 jendela 4-petak** (horizontal, vertikal, diagonal naik & turun):
  - `3 bidak AI + 1 kosong`: **+12**
  - `2 bidak AI + 2 kosong`: **+3**
  - `3 bidak Lawan + 1 kosong`: **-80** (penalti besar agar AI memprioritaskan blokir ancaman langsung)
  - `2 bidak Lawan + 2 kosong`: **-2**
- Memberikan **bonus kontrol kolom tengah (`kolom 3`)** sebesar **+4** per selisih bidak di kolom tengah.

In [ ]:
def evaluate_window(window, ai_piece):
    """Hitung skor heuristik satu jendela berisi 4 petak dari sudut pandang ai_piece."""
    opp_piece = HUMAN if ai_piece == AI else AI
    ai_count = window.count(ai_piece)
    opp_count = window.count(opp_piece)
    empty_count = window.count(EMPTY)

    if ai_count == 4:
        return EVAL_WEIGHTS["WIN"]
    if opp_count == 4:
        return -EVAL_WEIGHTS["WIN"]
    if ai_count == 3 and empty_count == 1:
        return EVAL_WEIGHTS["AI_THREE"]
    if ai_count == 2 and empty_count == 2:
        return EVAL_WEIGHTS["AI_TWO"]
    if opp_count == 3 and empty_count == 1:
        return EVAL_WEIGHTS["OPP_THREE"]
    if opp_count == 2 and empty_count == 2:
        return EVAL_WEIGHTS["OPP_TWO"]
    return 0


def analyze_board(board, ai_piece=AI):
    """Hitung total skor evaluasi papan beserta rincian fitur heuristiknya."""
    opp_piece = HUMAN if ai_piece == AI else AI
    center_col = COLS // 2

    center_ai = sum(1 for r in range(ROWS) if board[r][center_col] == ai_piece)
    center_opp = sum(1 for r in range(ROWS) if board[r][center_col] == opp_piece)

    ai_three = 0
    ai_two = 0
    opp_three = 0
    opp_two = 0
    window_score = 0

    def process_window(w):
        nonlocal ai_three, ai_two, opp_three, opp_two, window_score
        ai_c = w.count(ai_piece)
        opp_c = w.count(opp_piece)
        emp_c = w.count(EMPTY)
        if ai_c == 3 and emp_c == 1:
            ai_three += 1
        elif ai_c == 2 and emp_c == 2:
            ai_two += 1
        elif opp_c == 3 and emp_c == 1:
            opp_three += 1
        elif opp_c == 2 and emp_c == 2:
            opp_two += 1
        window_score += evaluate_window(w, ai_piece)

    # Horizontal
    for r in range(ROWS):
        for c in range(COLS - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]])

    # Vertikal
    for c in range(COLS):
        for r in range(ROWS - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]])

    # Diagonal turun (\)
    for r in range(ROWS - WINDOW_LENGTH + 1):
        for c in range(COLS - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]])

    # Diagonal naik (/)
    for r in range(WINDOW_LENGTH - 1, ROWS):
        for c in range(COLS - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]])

    center_score = (center_ai - center_opp) * EVAL_WEIGHTS["CENTER_PER_DISC"]
    return {
        "total_score": window_score + center_score,
        "center_ai": center_ai,
        "center_opp": center_opp,
        "center_score": center_score,
        "ai_three": ai_three,
        "ai_two": ai_two,
        "opp_three": opp_three,
        "opp_two": opp_two,
        "window_score": window_score,
    }


def evaluate_board(board, ai_piece=AI):
    """Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state."""
    return analyze_board(board, ai_piece)["total_score"]


# Uji fungsi evaluasi pada papan contoh
eval_demo = analyze_board(sample_board, AI)
print("Rincian Evaluasi Papan Contoh (Sudut Pandang AI):")
for k, v in eval_demo.items():
    print(f"  - {k:14s}: {v}")

---
## Bagian 4: Implementasi Depth-Limited Minimax & Alpha-Beta Pruning
Di bawah ini terdapat dua versi algoritma pencarian:
1. `minimax(...)`: **Depth-Limited Minimax Murni** (menjelajah seluruh cabang hingga batas `max_depth`).
2. `minimax_alpha_beta(...)`: **Depth-Limited Minimax + Alpha-Beta Pruning** (memangkas cabang saat `alpha >= beta`).

Keduanya menerapkan **penalti kedalaman** (`WIN_SCORE - depth` saat menang, `depth - WIN_SCORE` saat kalah) seperti pola pada Sesi 4 agar Bot AI memilih kemenangan tercepat dan menunda kekalahan selama mungkin.

In [ ]:
def minimax(board, depth, max_depth, is_maximizing, ai_piece, stats):
    """Depth-Limited Minimax Murni tanpa Alpha-Beta Pruning."""
    stats["nodes"] += 1
    opp_piece = HUMAN if ai_piece == AI else AI

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == opp_piece:
        return depth - WIN_SCORE
    if winner == "DRAW":
        return 0

    if depth >= max_depth:
        return evaluate_board(board, ai_piece)

    valid_cols = get_valid_locations(board)

    if is_maximizing:
        best_score = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = ai_piece
            score = minimax(board, depth + 1, max_depth, False, ai_piece, stats)
            board[row][col] = EMPTY
            if score > best_score:
                best_score = score
        return best_score

    best_score = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = opp_piece
        score = minimax(board, depth + 1, max_depth, True, ai_piece, stats)
        board[row][col] = EMPTY
        if score < best_score:
            best_score = score
    return best_score


def minimax_alpha_beta(board, depth, max_depth, alpha, beta, is_maximizing, ai_piece, stats):
    """Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning."""
    stats["nodes"] += 1
    opp_piece = HUMAN if ai_piece == AI else AI

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == opp_piece:
        return depth - WIN_SCORE
    if winner == "DRAW":
        return 0

    if depth >= max_depth:
        return evaluate_board(board, ai_piece)

    valid_cols = get_valid_locations(board)

    if is_maximizing:
        max_eval = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = ai_piece
            eval_score = minimax_alpha_beta(
                board, depth + 1, max_depth, alpha, beta, False, ai_piece, stats
            )
            board[row][col] = EMPTY
            if eval_score > max_eval:
                max_eval = eval_score
            if max_eval > alpha:
                alpha = max_eval
            if alpha >= beta:
                break  # Pemangkasan cabang MIN
        return max_eval

    min_eval = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = opp_piece
        eval_score = minimax_alpha_beta(
            board, depth + 1, max_depth, alpha, beta, True, ai_piece, stats
        )
        board[row][col] = EMPTY
        if eval_score < min_eval:
            min_eval = eval_score
        if min_eval < beta:
            beta = min_eval
        if alpha >= beta:
            break  # Pemangkasan cabang MAX
    return min_eval


def find_best_move(board, max_depth=4, ai_piece=AI, use_pruning=True):
    """Cari kolom terbaik beserta skor evaluasi tiap kolom dan jumlah node."""
    working_board = clone_board(board)
    valid_cols = get_valid_locations(working_board)
    column_scores = {c: None for c in range(COLS)}

    if not valid_cols:
        return {
            "best_col": -1,
            "best_score": 0,
            "column_scores": column_scores,
            "nodes": 0,
            "elapsed_ms": 0.0,
            "max_depth": max_depth,
            "use_pruning": use_pruning,
        }

    stats = {"nodes": 0}
    t0 = time.perf_counter()

    best_score = -math.inf
    best_col = valid_cols[0]
    alpha = -math.inf
    beta = math.inf

    for col in valid_cols:
        row = get_next_open_row(working_board, col)
        working_board[row][col] = ai_piece

        if use_pruning:
            score = minimax_alpha_beta(
                working_board, 1, max_depth, alpha, beta, False, ai_piece, stats
            )
        else:
            score = minimax(
                working_board, 1, max_depth, False, ai_piece, stats
            )

        working_board[row][col] = EMPTY
        column_scores[col] = score

        if score > best_score:
            best_score = score
            best_col = col
        if use_pruning and best_score > alpha:
            alpha = best_score

    t1 = time.perf_counter()
    return {
        "best_col": best_col,
        "best_score": best_score,
        "column_scores": column_scores,
        "nodes": stats["nodes"],
        "elapsed_ms": round((t1 - t0) * 1000, 2),
        "max_depth": max_depth,
        "use_pruning": use_pruning,
    }


def choose_move(board, difficulty="Hard", ai_piece=AI, use_pruning=None, custom_depth=None):
    """Pilih langkah Bot AI sesuai tingkat kesulitan ('Easy', 'Medium', 'Hard')."""
    preset = DIFFICULTY_PRESETS.get(difficulty, DIFFICULTY_PRESETS["Hard"])

    if difficulty == "Easy" and custom_depth is None:
        valid_cols = get_valid_locations(board)
        column_scores = {c: None for c in range(COLS)}
        if not valid_cols:
            return {
                "best_col": -1, "best_score": 0, "column_scores": column_scores,
                "nodes": 0, "elapsed_ms": 0.0, "max_depth": 0,
                "use_pruning": False, "is_random": True,
            }
        working_board = clone_board(board)
        t0 = time.perf_counter()
        for col in valid_cols:
            row = get_next_open_row(working_board, col)
            working_board[row][col] = ai_piece
            column_scores[col] = evaluate_board(working_board, ai_piece)
            working_board[row][col] = EMPTY
        chosen = random.choice(valid_cols)
        t1 = time.perf_counter()
        return {
            "best_col": chosen,
            "best_score": column_scores[chosen],
            "column_scores": column_scores,
            "nodes": len(valid_cols),
            "elapsed_ms": round((t1 - t0) * 1000, 2),
            "max_depth": 0,
            "use_pruning": False,
            "is_random": True,
        }

    max_depth = custom_depth if custom_depth is not None else preset["max_depth"]
    pruning = use_pruning if use_pruning is not None else preset["use_pruning"]
    res = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=pruning)
    res["is_random"] = False
    return res


def compare_algorithms(board, max_depth=4, ai_piece=AI):
    """Bandingkan Minimax Murni vs Alpha-Beta Pruning pada state papan yang sama."""
    pure = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=False)
    ab = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=True)
    reduction = round(((pure["nodes"] - ab["nodes"]) / pure["nodes"]) * 100, 1) if pure["nodes"] > 0 else 0.0
    return {
        "max_depth": max_depth,
        "pure": pure,
        "alphabeta": ab,
        "reduction_pct": reduction,
        "same_decision": pure["best_col"] == ab["best_col"],
    }

---
## Bagian 5: Pengujian Terstruktur & Benchmark Efisiensi Node
Sel-sel di bawah ini berjalan otomatis (tanpa `input()`) untuk menguji:
1. Ketepatan taktis Bot AI (menang 1 langkah, memblokir lawan, memilih kolom tengah di awal).
2. Perbandingan jumlah *node* yang dievaluasi dan waktu komputasi **Minimax Murni vs Alpha-Beta Pruning** pada berbagai kedalaman (`depth = 1..4`).
3. Simulasi pertandingan antar tingkat kesulitan (5 game per pasangan, mengikuti standar uji coba Sesi 4).

In [ ]:
# 1. Pengujian Taktis Unit Test
def run_tactical_tests():
    # Kasus A: Papan kosong -> harus pilih kolom tengah (3)
    b_empty = create_board()
    res_empty = find_best_move(b_empty, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_empty["best_col"] == 3, f"Harusnya kolom 3, dapat {res_empty['best_col']}"

    # Kasus B: AI punya 3 beruntun (kolom 0,1,2 di baris 5) -> wajib selesaikan menang di kolom 3
    b_win = create_board()
    for c in [0, 1, 2]:
        drop_piece(b_win, 5, c, AI)
    for c in [0, 1]:
        drop_piece(b_win, 4, c, HUMAN)
    res_win = find_best_move(b_win, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_win["best_col"] == 3 and res_win["best_score"] == WIN_SCORE - 1

    # Kasus C: Manusia punya 3 beruntun (kolom 4,5,6 di baris 5) -> Bot wajib blokir di kolom 3
    b_block = create_board()
    for c in [4, 5, 6]:
        drop_piece(b_block, 5, c, HUMAN)
    drop_piece(b_block, 5, 0, AI)
    drop_piece(b_block, 4, 0, AI)
    res_block = find_best_move(b_block, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_block["best_col"] == 3, f"Harusnya blokir kolom 3, dapat {res_block['best_col']}"

    # Kasus D: Keputusan Minimax Murni dan Alpha-Beta harus 100% identik
    comp = compare_algorithms(b_block, max_depth=4, ai_piece=AI)
    assert comp["same_decision"], "Keputusan Minimax vs Alpha-Beta berbeda!"

    print("[PASS] Semua 4 pengujian taktis berhasil 100%!")
    print(f"  - Papan kosong -> Kol {res_empty['best_col']} (skor {res_empty['best_score']})")
    print(f"  - Deteksi menang 1 langkah -> Kol {res_win['best_col']} (skor {res_win['best_score']})")
    print(f"  - Blokir ancaman lawan     -> Kol {res_block['best_col']} (skor {res_block['best_score']})")
    print(f"  - Konsistensi Minimax vs Alpha-Beta (d=4): Kol {comp['pure']['best_col']} == Kol {comp['alphabeta']['best_col']} "
          f"(hemat {comp['reduction_pct']}% node)")


run_tactical_tests()

In [ ]:
# 2. Benchmark Minimax Murni vs Alpha-Beta Pruning (Depth 1 s.d. 4)
bench_board = create_board()
sample_opening = [(3, HUMAN), (3, AI), (2, HUMAN), (4, AI), (2, HUMAN), (3, AI)]
for col_m, piece_m in sample_opening:
    drop_piece(bench_board, get_next_open_row(bench_board, col_m), col_m, piece_m)

print("Posisi Papan Benchmark (6 langkah pembuka):")
print_board(bench_board)

rows_bench = []
for d in [1, 2, 3, 4]:
    comp = compare_algorithms(bench_board, max_depth=d, ai_piece=AI)
    rows_bench.append({
        "Max Depth": d,
        "Kolom Dipilih": comp["alphabeta"]["best_col"],
        "Skor Eval": comp["alphabeta"]["best_score"],
        "Node Minimax": comp["pure"]["nodes"],
        "Node Alpha-Beta": comp["alphabeta"]["nodes"],
        "Penghematan Node (%)": comp["reduction_pct"],
        "Waktu Minimax (ms)": comp["pure"]["elapsed_ms"],
        "Waktu Alpha-Beta (ms)": comp["alphabeta"]["elapsed_ms"],
        "Keputusan Sama?": comp["same_decision"],
    })

df_bench = pd.DataFrame(rows_bench)
print("\n=== TABEL BENCHMARK EFISIENSI NODE ===")
print(df_bench.to_string(index=False))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    depths = df_bench["Max Depth"]

    axes[0].bar(depths - 0.18, df_bench["Node Minimax"], width=0.35, label="Minimax Murni", color="#dc2626")
    axes[0].bar(depths + 0.18, df_bench["Node Alpha-Beta"], width=0.35, label="Alpha-Beta Pruning", color="#1d4ed8")
    axes[0].set_title("Perbandingan Jumlah Node Dievaluasi")
    axes[0].set_xlabel("Max Depth")
    axes[0].set_ylabel("Jumlah Node")
    axes[0].set_xticks(depths)
    axes[0].legend()
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    axes[1].bar(depths, df_bench["Penghematan Node (%)"], width=0.45, color="#15803d")
    axes[1].set_title("Persentase Penghematan Node Alpha-Beta")
    axes[1].set_xlabel("Max Depth")
    axes[1].set_ylabel("Penghematan (%)")
    axes[1].set_ylim(0, 100)
    axes[1].set_xticks(depths)
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Simulasi Turnamen Otomatis Bot vs Bot (5 Pertandingan per Skenario)
def simulate_match(p1_cfg, p2_cfg, seed=None, opening=None):
    if seed is not None:
        random.seed(seed)
    board = create_board()
    moves = 0
    nodes_p1, nodes_p2 = [], []

    if opening:
        for idx, col in enumerate(opening):
            piece = HUMAN if idx % 2 == 0 else AI
            drop_piece(board, get_next_open_row(board, col), col, piece)
            moves += 1
        turn = HUMAN if len(opening) % 2 == 0 else AI
    else:
        turn = HUMAN  # HUMAN = Pemain 1, AI = Pemain 2

    while True:
        cfg = p1_cfg if turn == HUMAN else p2_cfg
        res = choose_move(
            board,
            difficulty=cfg["difficulty"],
            ai_piece=turn,
            use_pruning=cfg.get("use_pruning"),
            custom_depth=cfg.get("custom_depth"),
        )
        col = res["best_col"]
        drop_piece(board, get_next_open_row(board, col), col, turn)
        moves += 1
        if turn == HUMAN:
            nodes_p1.append(res["nodes"])
        else:
            nodes_p2.append(res["nodes"])

        winner = check_winner(board)
        if winner is not None:
            return {
                "winner": winner,
                "moves": moves,
                "avg_nodes_p1": round(sum(nodes_p1) / max(1, len(nodes_p1)), 1),
                "avg_nodes_p2": round(sum(nodes_p2) / max(1, len(nodes_p2)), 1),
            }
        turn = AI if turn == HUMAN else HUMAN


varied_openings = [(3, 3), (2, 3), (4, 3), (3, 2), (3, 4)]

scenarios = [
    ("Easy (Random) vs Medium (Depth 3 Minimax)", {"difficulty": "Easy"}, {"difficulty": "Medium"}, None),
    ("Easy (Random) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Easy"}, {"difficulty": "Hard"}, None),
    ("Medium (Depth 3) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Medium"}, {"difficulty": "Hard"}, varied_openings),
]

summary_rows = []
for name, p1_c, p2_c, openings in scenarios:
    p1_wins = p2_wins = draws = 0
    total_moves = []
    avg_n2 = []
    for idx in range(5):
        op = openings[idx] if openings else None
        out = simulate_match(p1_c, p2_c, seed=100 + idx, opening=op)
        total_moves.append(out["moves"])
        avg_n2.append(out["avg_nodes_p2"])
        if out["winner"] == HUMAN:
            p1_wins += 1
        elif out["winner"] == AI:
            p2_wins += 1
        else:
            draws += 1

    summary_rows.append({
        "Skenario (P1 vs P2)": name,
        "P1 Menang": p1_wins,
        "P2 Menang": p2_wins,
        "Seri": draws,
        "Rata-rata Langkah": round(sum(total_moves) / len(total_moves), 1),
        "Rata-rata Node P2": round(sum(avg_n2) / len(avg_n2), 1),
    })

df_sim = pd.DataFrame(summary_rows)
print("=== HASIL SIMULASI 5 GAME PER SKENARIO ===")
print(df_sim.to_string(index=False))

---
## Bagian 6: Arena Permainan Interaktif Manusia vs Bot AI (Google Colab)
Untuk mencoba bermain langsung di Google Colab, ubah nilai `JALANKAN_GAME_INTERAKTIF = True` pada sel di bawah lalu jalankan sel tersebut.
Selama bermain, arena akan menampilkan:
- Papan Connect-Four terkini (`O` = Manusia, `X` = Bot AI)
- Skor evaluasi tiap kolom `[0..6]` dari sudut pandang Bot AI
- Jumlah node yang dievaluasi dan waktu komputasi per langkah Bot.

In [ ]:
def play_game(difficulty="Hard", human_first=True, custom_depth=None, use_pruning=None):
    """Arena interaktif Manusia ('O') vs Bot AI ('X') di Google Colab."""
    board = create_board()
    current_turn = HUMAN if human_first else AI
    step = 0

    print("=== ARENA INTERAKTIF CONNECT-FOUR AI (TIM 12) ===")
    print(f"Tingkat Kesulitan : {difficulty}")
    print("Simbol Pemain     : Manusia = 'O' | Bot AI = 'X'")
    print("Masukkan nomor kolom 0 s.d. 6 (atau 'q' untuk keluar).\n")

    while True:
        print_board(board)
        print(f"Skor Evaluasi Papan Statis (sudut pandang AI): {evaluate_board(board, AI)}")

        if current_turn == HUMAN:
            valid = [c for c in range(COLS) if is_valid_location(board, c)]
            while True:
                raw = input(f"[Giliran Manusia 'O'] Pilih kolom {valid} (atau 'q'): ").strip()
                if raw.lower() == "q":
                    print("Permainan dihentikan oleh pemain.")
                    return
                if raw.isdigit() and int(raw) in valid:
                    col = int(raw)
                    break
                print("Input tidak valid! Pilih kolom yang masih tersedia.")

            row = get_next_open_row(board, col)
            drop_piece(board, row, col, HUMAN)
            step += 1
            print(f"-> Manusia menjatuhkan bidak di Kolom {col}, Baris {row}.\n")
        else:
            print("[Giliran Bot AI 'X'] Menghitung langkah terbaik...")
            res = choose_move(
                board,
                difficulty=difficulty,
                ai_piece=AI,
                use_pruning=use_pruning,
                custom_depth=custom_depth,
            )
            col = res["best_col"]
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, AI)
            step += 1

            scores_str = " | ".join(
                f"K{c}:{res['column_scores'][c]}" if res["column_scores"][c] is not None else f"K{c}:Penuh"
                for c in range(COLS)
            )
            print(f"-> Bot memilih Kolom {col} (Baris {row}) | Skor: {res['best_score']} "
                  f"| Node: {res['nodes']} | Waktu: {res['elapsed_ms']} ms")
            print(f"   Skor Evaluasi Kolom: [{scores_str}]\n")

        winner, win_cells = get_winner_info(board)
        if winner is not None:
            print_board(board)
            if winner == "DRAW":
                print(f"=== HASIL AKHIR: SERI setelah {step} langkah! ===")
            elif winner == HUMAN:
                print(f"=== SELAMAT! Manusia ('O') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            else:
                print(f"=== BOT AI ('X') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            break

        current_turn = AI if current_turn == HUMAN else HUMAN


# Ubah menjadi True jika ingin langsung bermain interaktif di Google Colab
JALANKAN_GAME_INTERAKTIF = False

if JALANKAN_GAME_INTERAKTIF:
    play_game(difficulty="Hard", human_first=True)
else:
    print("Mode Run All aman: set JALANKAN_GAME_INTERAKTIF = True atau panggil play_game() untuk bermain.")